# Otimizadores

Neste notebook vamos estudar algoritmos de otimização para treinar redes neurais. Eles buscam **minimizar a função de custo** ajustando os parâmetros do modelo de forma iterativa.
Também abordaremos a utilização do pacote `torch.optim`.

## Fundamentos

Os algoritmos de otimização são essenciais para treinar redes neurais. Eles buscam **minimizar a função de custo** $L(\theta)$ ajustando os parâmetros $\theta$ do modelo de forma iterativa.

### Gradiente Descendente Básico
Fornece um passo uniforme em todas as direções, sem considerar a escala ou correlação entre dimensões, o que pode gerar trajetórias lentas em funções mal condicionadas. O algoritmo atualiza os parâmetros na direção oposta ao gradiente:

$$\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t)$$

- $\theta_t$: parâmetros atuais do modelo  
- $\eta$: taxa de aprendizado (tamanho do passo)  
- $\nabla L(\theta_t)$: gradiente da função de custo em relação aos parâmetros  

### Momentum
Reduz oscilações em direções de alta curvatura e acelera a convergência em vales alongados, aproximando-se de um método de aceleração de gradiente. Ele adiciona uma memória das atualizações anteriores:

$$v_{t+1} = \beta v_t + \nabla L(\theta_t)$$
$$\theta_{t+1} = \theta_t - \eta v_{t+1}$$

- $v_t$: vetor de "velocidade" acumulada  
- $\beta$: coeficiente de momentum (0.9 é comum)  
- $\eta$: taxa de aprendizado  

### AdaGrad
Normaliza o passo por dimensão, aumentando estabilidade em problemas esparsos, mas a acumulação crescente de $G_t$ tende a reduzir excessivamente a taxa de aprendizado ao longo do tempo. Sua atualização é dada por:

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t + \epsilon}} \nabla L(\theta_t)$$

- $G_t$: soma acumulada dos quadrados dos gradientes  
- $\epsilon$: pequeno valor para estabilidade numérica  
- $\eta$: taxa de aprendizado inicial  

### RMSprop
Evita a saturação do AdaGrad ao introduzir esquecimento exponencial, permitindo adaptação contínua da taxa de aprendizado em problemas não estacionários. A regra de atualização é:

$$v_t = \beta v_{t-1} + (1-\beta)(\nabla L(\theta_t))^2$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{v_t + \epsilon}} \nabla L(\theta_t)$$

- $v_t$: média móvel dos gradientes ao quadrado  
- $\beta$: fator de decaimento (tipicamente 0.9)  
- $\epsilon$: termo de estabilidade  
- $\eta$: taxa de aprendizado  

### Adam
Fornece estimativas corrigidas de primeira e segunda ordem, resultando em passos adaptativos que combinam aceleração e escalonamento por dimensão, o que o torna robusto em larga escala e em arquiteturas profundas. O procedimento é:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\nabla L(\theta_t)$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)(\nabla L(\theta_t))^2$$

Correções de viés (necessárias pois $m_t$ e $v_t$ tendem a começar enviesados para baixo no início):

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t} \quad , \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

Atualização final dos parâmetros:

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon}\,\hat{m}_t$$

- $m_t$: estimativa de primeira ordem (média móvel dos gradientes)  
- $v_t$: estimativa de segunda ordem (média móvel dos gradientes ao quadrado)  
- $\hat{m}_t, \hat{v}_t$: versões corrigidas de viés  
- $\beta_1$: fator de decaimento para $m_t$ (ex: 0.9)  
- $\beta_2$: fator de decaimento para $v_t$ (ex: 0.999)  
- $\epsilon$: termo de estabilidade numérica  
- $\eta$: taxa de aprendizado


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
from matplotlib import pyplot as plt
import numpy as np
from tqdm import tqdm
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Exemplo: Função de Himmelblau

Para analisar e comparar diferentes otimizadores, utilizamos a **função de Himmelblau**, uma função clássica em otimização não convexa. Ela é definida como:

$$
f(x, y) = (x^2 + y - 11)^2 + (x + y^2 - 7)^2
$$

- Possui **quatro mínimos locais**, sendo todos também mínimos globais, além de pontos de sela.  
- É frequentemente usada como benchmark porque apresenta um relevo complexo, que desafia os algoritmos de otimização.  

In [ ]:
def himmelblau(x):
    x1, x2 = x[0], x[1]
    return (x1**2 + x2 - 11)**2 + (x1 + x2**2 - 7)**2

In [ ]:
import plotly.graph_objects as go

X = np.linspace(-6, 6, 200)
Y = np.linspace(-6, 6, 200)
X, Y = np.meshgrid(X, Y)
Z = himmelblau([X, Y])

fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y, colorscale="viridis")])
fig.update_layout(
    title="Himmelblau's Function",
    scene=dict(
        xaxis_title="x1",
        yaxis_title="x2",
        zaxis_title="f(x1, x2)"
    ),
    width=800,
    height=600
)
fig.show()

### Otimização em Funções Não Convexas

Considere uma função objetivo $f: \mathbb{R}^2 \to \mathbb{R}$ como a função de Himmelblau:

$$
f(x_1, x_2) = (x_1^2 + x_2 - 11)^2 + (x_1 + x_2^2 - 7)^2
$$

Esse tipo de função apresenta múltiplos mínimos locais e pontos de sela, o que a torna um bom cenário para comparar otimizadores.

O problema geral de otimização pode ser formulado como:

$$
\min_{x \in \mathbb{R}^n} f(x)
$$

onde $x = (x_1, x_2, \dots, x_n)$ são os parâmetros que queremos ajustar.

Durante o processo iterativo, mantemos uma sequência de pontos $\{x_t\}$, atualizados de acordo com uma regra específica definida por cada otimizador. Em geral:

$$
x_{t+1} = x_t - \Delta_t
$$

- $x_t$: vetor de parâmetros na iteração $t$  
- $\Delta_t$: passo de atualização definido pelo algoritmo  

### Utilizando o `torch.optim`

O PyTorch fornece o módulo `torch.optim`, que implementa algoritmos de otimização responsáveis por atualizar os parâmetros do modelo a partir dos gradientes. Por exemplo, para utilizar o otimizador Adam, fazemos:

```python
optimizer = optim.Adam(model.parameters(), lr=0.001)
```

#### Parâmetros

Esses otimizadores não “sabem” automaticamente o que deve ser atualizado. Por isso, é necessário passar explicitamente `model.parameters()`, que é um iterável contendo todos os tensores treináveis do modelo. Esses tensores têm `requires_grad=True` e, após a chamada de `backward()`, armazenam seus gradientes em `param.grad`. O otimizador apenas percorre esses parâmetros e aplica uma regra de atualização baseada nesses gradientes. Conceitualmente, o que acontece é algo como:

```python
for param in model.parameters():
    param = param - lr * param.grad
```

#### Atualização

Um detalhe importante é que, no PyTorch, os gradientes **acumulam por padrão**. Ou seja, se você não limpar os gradientes antes de uma nova iteração, eles serão somados aos anteriores. Por isso, usamos:

```python
optimizer.zero_grad()
```

Esse comando zera os gradientes de todos os parâmetros registrados no otimizador, garantindo que cada passo utilize apenas os gradientes da iteração atual. Após calcular a loss e chamar `backward()`, usamos:

```python
optimizer.step()
```

Esse método aplica a regra de atualização do algoritmo escolhido (SGD, Adam, etc.) sobre cada parâmetro, utilizando os gradientes armazenados em `param.grad`.

In [ ]:
def run_optimizer(opt_class, opt_params, x_init, steps=50, lr=0.01):
    x = x_init.clone().detach().requires_grad_(True)
    optimizer = opt_class([x], lr=lr, **opt_params)
    track, losses = [], []
    for _ in range(steps):
        optimizer.zero_grad()
        loss = himmelblau(x)
        loss.backward()
        optimizer.step()
        track.append(x.detach().clone())
        losses.append(loss.item())
    return track, losses

def plot_progress(track, losses, func, title=""):
    fig, (ax1, ax2) = plt.subplots(1,2, figsize=(12, 5))
    
    X = np.linspace(-6, 6, 400)
    Y = np.linspace(-6, 6, 400)
    X, Y = np.meshgrid(X, Y)
    Z = (X**2 + Y - 11)**2 + (X + Y**2 - 7)**2
    ax1.contour(X, Y, Z, levels=np.logspace(0, 5, 35), cmap='viridis')

    track = torch.stack(track).t()
    ax1.plot(track[0,:], track[1,:], marker='o', color="red")
    ax1.scatter(track[0,0], track[1,0], color='yellow', marker='*', s=200, label="start")
    ax1.set_title(f'progress of x ({title})')
    ax1.set_xlim(-6, 6)
    ax1.set_ylim(-6, 6)
    ax1.set_xlabel("x1")
    ax1.set_ylabel("x2")
    ax1.legend()

    # Loss curve
    ax2.set_title(f'progress of f(x) ({title})')
    ax2.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    ax2.plot(range(len(losses)), losses, marker='o')
    ax2.set_ylabel('objective')
    ax2.set_xlabel('iteration')
    ax2.set_yscale("log")
    
    plt.show()

In [ ]:
optimizers = {
    "SGD": (optim.SGD, {}),
    "SGD+Momentum": (optim.SGD, {"momentum": 0.2}),
    "Adagrad": (optim.Adagrad, {}),
    "RMSprop": (optim.RMSprop, {"alpha": 0.9}),
    "Adam": (optim.Adam, {})
}

x_start = torch.tensor([-4.0, 0.0])

for name, (opt_class, opt_params) in optimizers.items():
    track, losses = run_optimizer(opt_class, opt_params, x_start, steps=100, lr=0.03)
    plot_progress(track, losses, himmelblau, title=name)

## Exemplo: MNIST

Agora reunimos em um único modelo os principais elementos discutidos e comparamos diferentes otimizadores.

In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

# Transformação
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Datasets
train_data = torchvision.datasets.MNIST("./data", train=True, download=True, transform=transform)
test_data  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=transform)

# Subsets menores
train_subset = Subset(train_data, torch.randperm(len(train_data))[:3000])
val_subset   = Subset(test_data,  torch.randperm(len(test_data))[:1000])

# DataLoaders
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_subset, batch_size=64)

print(len(train_subset), len(val_subset))

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )
        
    def forward(self, x):
        x = self.layers(x)
        return x

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, epochs=10, device="cpu"):
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    model = model.to(device)

    for epoch in range(epochs):
        # ---- Treino ----
        model.train()
        total_loss, correct, total = 0, 0, 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(output, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
        
        history["train_loss"].append(total_loss / len(train_loader))
        history["train_acc"].append(100 * correct / total)

        # ---- Validação ----
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)

                output = model(data)
                loss = criterion(output, target)
                val_loss += loss.item()
                _, predicted = torch.max(output, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
        
        history["val_loss"].append(val_loss / len(val_loader))
        history["val_acc"].append(100 * correct / total)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {history['train_loss'][-1]:.4f}, "
              f"Train Acc: {history['train_acc'][-1]:.2f}% | "
              f"Val Loss: {history['val_loss'][-1]:.4f}, "
              f"Val Acc: {history['val_acc'][-1]:.2f}%")

    return history

In [ ]:
optimizers_config = {
    'SGD': {'lr': 0.01},
    'SGD+Momentum': {'lr': 0.01, 'momentum': 0.9},
    'Adagrad': {'lr': 0.01},
    'RMSprop': {'lr': 0.01},
    'Adam': {'lr': 0.01}
}

results = {}

for name, config in optimizers_config.items():
    print(f"\nTreinando com {name}...")

    # Novo modelo a cada otimizador
    model = SimpleNet().to(device)

    # Instanciar otimizador
    optimizer_class = getattr(optim, name.split('+')[0])
    optimizer = optimizer_class(model.parameters(), **config)
    print(optimizer)

    # Treinar e guardar histórico
    history = train_model(model, train_loader, val_loader, optimizer, epochs=10, device=device)
    results[name] = history

In [ ]:
plt.figure(figsize=(12,5))
colors = sns.color_palette("husl", len(results))

# Loss
plt.subplot(1,2,1)
for i, (name, hist) in enumerate(results.items()):
    color = colors[i]
    plt.plot(hist["train_loss"], label=f"{name} Train", color=color)
    plt.plot(hist["val_loss"], linestyle="--", label=f"{name} Val", color=color)
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Accuracy
plt.subplot(1,2,2)
for i, (name, hist) in enumerate(results.items()):
    color = colors[i]
    plt.plot(hist["train_acc"], label=f"{name} Train", color=color)
    plt.plot(hist["val_acc"], linestyle="--", label=f"{name} Val", color=color)
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()

plt.tight_layout()
plt.show()